# 01 — SmarTRIZ QLoRA SFT Training

Trains Qwen2.5-7B-Instruct (or 14B) with QLoRA + CoT loss masking.

**Prerequisites:** Run Notebook 00 first to produce `training_dataset_clean.json` and `test_split.json`.

In [ ]:
# ─── CONFIG — edit this cell only ────────────────────────
MODEL_SIZE = '7b'          # change to '14b' for 14B training
DRIVE_PATH = '/content/drive/MyDrive/smartriz/'
WANDB_PROJECT = 'smartriz-sft'

MODEL_NAME   = f'Qwen/Qwen2.5-{MODEL_SIZE.upper()}-Instruct'
DATASET_PATH = f'{DRIVE_PATH}data/training_dataset_clean.json'
OUTPUT_DIR   = f'{DRIVE_PATH}checkpoints/sft-{MODEL_SIZE}/'

LORA_R     = 16 if MODEL_SIZE == '7b' else 32
LORA_ALPHA = 32 if MODEL_SIZE == '7b' else 64
BATCH_SIZE = 4  if MODEL_SIZE == '7b' else 2
GRAD_ACCUM = 4  if MODEL_SIZE == '7b' else 8
MAX_SEQ_LEN   = 3072
LEARNING_RATE = 2e-4
NUM_EPOCHS    = 3
SAVE_STEPS    = 200
# ──────────────────────────────────────────────────────────

In [ ]:
!pip install -q transformers==4.45.2 peft==0.13.2 trl==0.11.4 bitsandbytes==0.44.1 accelerate==0.34.2 datasets==3.0.1 wandb tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── Resume from checkpoint detection
from pathlib import Path

checkpoint_dirs = sorted(Path(OUTPUT_DIR).glob('checkpoint-*')) if Path(OUTPUT_DIR).exists() else []
RESUME_FROM = str(checkpoint_dirs[-1]) if checkpoint_dirs else None
if RESUME_FROM:
    print(f'Resuming from checkpoint: {RESUME_FROM}')
else:
    print('No checkpoint found — training from scratch')

In [ ]:
import json

def load_training(path):
    with open(path) as f:
        data = json.load(f)
    return data['cases'] if isinstance(data, dict) else data

cases = load_training(DATASET_PATH)
print(f'Loaded {len(cases)} training examples')

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

SYSTEM_PROMPT = (
    'You are SmarTRIZ, an expert engineering innovation assistant. '
    'Solve technical problems using TRIZ methodology. Identify the '
    'technical contradiction, select inventive principles from the '
    'Altshuller matrix, reason step by step, and propose a solution.'
)

def format_example(case):
    cp = case['contradiction_pair']
    principles = ', '.join(case['inventive_principles'])
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': case['problem']},
        {'role': 'assistant', 'content': (
            f'<think>\n{case["reasoning_chain"]}\n</think>\n'
            f'Contradiction:\n\n'
            f'Improving: {cp["improving_parameter"]}\n'
            f'Worsening: {cp["worsening_parameter"]}\n\n'
            f'Inventive Principles: {principles}\n'
            f'Solution:\n{case["solution"]}'
        )}
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

sample = format_example(cases[0])
print('Sample formatted example (first 600 chars):')
print(sample[:600])

In [ ]:
# ── Custom CoT loss masking collator
# Masks labels for system+user turns and <think>/<\/think> tags themselves.
# Keeps loss on reasoning inside <think> and on structured output after <\/think>.
import torch
from dataclasses import dataclass
from typing import Any

@dataclass
class CotMaskingCollator:
    tokenizer: Any
    max_length: int = 3072

    def _tok(self, text):
        return self.tokenizer.encode(text, add_special_tokens=False)

    def _find_sublist(self, lst, sub):
        n = len(sub)
        for i in range(len(lst) - n + 1):
            if lst[i:i+n] == sub:
                return i
        return -1

    def __call__(self, features):
        im_start   = self._tok('<|im_start|>')
        im_end     = self._tok('<|im_end|>')
        user_tok   = self._tok('user')
        system_tok = self._tok('system')
        asst_tok   = self._tok('assistant')
        think_open  = self._tok('<think>')
        think_close = self._tok('</think>')

        batch_input_ids, batch_labels = [], []

        for f in features:
            ids = list(f['input_ids'][:self.max_length])
            labels = list(ids)
            i = 0
            while i < len(ids):
                if ids[i:i+len(im_start)] == im_start:
                    rs = i + len(im_start)
                    if ids[rs:rs+len(system_tok)] == system_tok or ids[rs:rs+len(user_tok)] == user_tok:
                        # Mask entire system/user turn
                        end = self._find_sublist(ids[i:], im_end)
                        if end != -1:
                            for j in range(i, i + end + len(im_end)):
                                if j < len(labels):
                                    labels[j] = -100
                    elif ids[rs:rs+len(asst_tok)] == asst_tok:
                        # Mask <|im_start|>assistant\n prefix
                        content_start = rs + len(asst_tok) + 1
                        for j in range(i, min(content_start, len(labels))):
                            labels[j] = -100
                        # Mask <think> tag itself
                        tp = self._find_sublist(ids[content_start:], think_open)
                        if tp != -1:
                            for j in range(content_start + tp, content_start + tp + len(think_open)):
                                if j < len(labels): labels[j] = -100
                        # Mask </think> tag itself
                        cp = self._find_sublist(ids[content_start:], think_close)
                        if cp != -1:
                            for j in range(content_start + cp, content_start + cp + len(think_close)):
                                if j < len(labels): labels[j] = -100
                i += 1

            batch_input_ids.append(ids)
            batch_labels.append(labels)

        max_len = max(len(x) for x in batch_input_ids)
        pad_id  = self.tokenizer.pad_token_id or 0
        input_ids = torch.tensor([x + [pad_id] * (max_len - len(x)) for x in batch_input_ids])
        labels_t  = torch.tensor([x + [-100]  * (max_len - len(x)) for x in batch_labels])
        attn_mask = (input_ids != pad_id).long()
        return {'input_ids': input_ids, 'attention_mask': attn_mask, 'labels': labels_t}

print('CotMaskingCollator defined')

In [ ]:
# ── Verify masking on one example
sample_encoded = tokenizer(format_example(cases[0]), return_tensors='pt',
                           truncation=True, max_length=MAX_SEQ_LEN)
collator = CotMaskingCollator(tokenizer=tokenizer, max_length=MAX_SEQ_LEN)
batch = collator([{'input_ids': sample_encoded['input_ids'][0].tolist()}])

ids  = batch['input_ids'][0]
lbls = batch['labels'][0]

print('=== Masking verification (✓=loss computed, ✗=masked) — first 150 tokens ===')
for k in range(min(150, len(ids))):
    tok_str = tokenizer.decode([ids[k].item()], skip_special_tokens=False)
    mark = '✓' if lbls[k].item() != -100 else '✗'
    print(f'{mark}{tok_str!r}', end=' ')
print('\n...(truncated)')

In [ ]:
# ── Load base model in 4-bit NF4
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model.config.use_cache = False

lora_cfg = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

In [ ]:
# ── Prepare HF Dataset with proper held-out eval split
import warnings
import json
from pathlib import Path
from datasets import Dataset as HFDataset
from tqdm.auto import tqdm

TEST_SPLIT_PATH = f'{DRIVE_PATH}data/test_split.json'

formatted = [{'text': format_example(c)} for c in tqdm(cases, desc='formatting train')]

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_SEQ_LEN, padding=False)

hf_train = HFDataset.from_list(formatted)
hf_train = hf_train.map(tokenize, batched=True, remove_columns=['text'])

# Load proper held-out eval set created by Notebook 00
if Path(TEST_SPLIT_PATH).exists():
    with open(TEST_SPLIT_PATH) as f:
        test_cases_raw = json.load(f)['cases']
    formatted_eval = [{'text': format_example(c)} for c in tqdm(test_cases_raw, desc='formatting eval')]
    hf_eval = HFDataset.from_list(formatted_eval)
    hf_eval = hf_eval.map(tokenize, batched=True, remove_columns=['text'])
    print(f'Loaded held-out eval set: {len(hf_eval)} examples from test_split.json')
else:
    warnings.warn(
        '\n\n⚠️  test_split.json not found — falling back to 5% of training data as eval.\n'
        '   Run Notebook 00 first to create a proper held-out eval set.\n',
        UserWarning, stacklevel=2
    )
    split = hf_train.train_test_split(test_size=0.05, seed=42)
    hf_train = split['train']
    hf_eval  = split['test']
    print(f'WARNING: Using internal eval split ({len(hf_eval)} examples) — NOT held-out data')

print(f'Train: {len(hf_train)} | Eval: {len(hf_eval)}')

In [ ]:
# ── Train with SFTTrainer + custom collator
import wandb
from transformers import TrainingArguments
from trl import SFTTrainer

wandb.init(project=WANDB_PROJECT, name=f'sft-{MODEL_SIZE}', resume='allow')

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    lr_scheduler_type='cosine',
    warmup_steps=50,
    bf16=True,
    fp16=False,
    gradient_checkpointing=True,
    logging_steps=10,
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    evaluation_strategy='steps',
    eval_steps=SAVE_STEPS,
    report_to='wandb',
    run_name=f'sft-{MODEL_SIZE}',
    optim='paged_adamw_8bit',
    dataloader_num_workers=2,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=hf_train,
    eval_dataset=hf_eval,
    data_collator=CotMaskingCollator(tokenizer=tokenizer, max_length=MAX_SEQ_LEN),
    tokenizer=tokenizer,
    max_seq_length=MAX_SEQ_LEN,
)

trainer.train(resume_from_checkpoint=RESUME_FROM)
print('Training complete')

In [ ]:
# ── Merge LoRA into base model and save
from pathlib import Path

merged_dir = OUTPUT_DIR + 'merged/'
if not Path(merged_dir + 'config.json').exists():
    print('Merging LoRA weights into base model...')
    model = model.merge_and_unload()
    model.save_pretrained(merged_dir)
    tokenizer.save_pretrained(merged_dir)
    print(f'Merged model saved to {merged_dir}')
else:
    print(f'Merged model already exists at {merged_dir} — skipping')

In [ ]:
# ── Quick eval: 5 held-out examples (expected vs generated)
import json
from transformers import pipeline

with open(f'{DRIVE_PATH}data/test_split.json') as f:
    test_cases = json.load(f)['cases'][:5]

gen_pipe = pipeline('text-generation', model=merged_dir, tokenizer=merged_dir,
                    device_map='auto', max_new_tokens=1024, do_sample=False)

for i, case in enumerate(test_cases):
    prompt = tokenizer.apply_chat_template(
        [{'role': 'system', 'content': SYSTEM_PROMPT},
         {'role': 'user', 'content': case['problem']}],
        tokenize=False, add_generation_prompt=True
    )
    result = gen_pipe(prompt)[0]['generated_text']
    generated = result[len(prompt):].strip()
    print(f'=== Example {i+1} ===')
    print(f'EXPECTED SOLUTION (first 300 chars):\n{case["solution"][:300]}...')
    print(f'GENERATED (first 300 chars):\n{generated[:300]}...')
    print()

print('Ready for Notebook 02 (DPO training)')